# Alarm Episode Clustering & Control Actions Export

Generate clustered alarm episodes and associated control actions for any target tag.

**Outputs**:
- Excel file with two sheets: `alarm_clusters` (episodes with cluster info) and `control_actions` (CHANGE events per cluster window)

**Configuration**: Edit the config cell below to switch target tags.

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — Change these to generate for a different tag
# ═══════════════════════════════════════════════════════════════════════════════

# Target tag and alarm type
TARGET_TAG = '03LIC_1016'           # Tag source name in events data
ALARM_CONDITION = 'PVLO'            # ConditionName to filter (PVLO, PVHI, etc.)

# Events data file (must contain VT_Start, Source, ConditionName, Action, etc.)
EVENTS_FILE = '/home/h604827/ControlActions/DATA/25_tags_events_preprocessed/51fda50f-24a3-4632-9803-7ec34744c34b.parquet'

# Trip data for filtering
TRIP_FILE = '../DATA/Final_List_Trip_Duration.csv'
FILTER_TRIPS = True  # Set False to skip trip filtering

# Clustering parameters
CLUSTER_GAP_THRESHOLD = 30          # Minutes gap to consider alarms as same cluster
ACTION_WINDOW_BEFORE = 240          # Minutes before cluster to capture control actions
ACTION_WINDOW_AFTER = 60            # Minutes after cluster to capture control actions
START_DATE = '2022-01-01'           # Only consider events from this date onwards

# Output — single folder per tag under RESULTS/
import os
RESULTS_DIR = f'../RESULTS/{TARGET_TAG}_episodes'
os.makedirs(RESULTS_DIR, exist_ok=True)
OUTPUT_FILE = f'{RESULTS_DIR}/{TARGET_TAG}_{ALARM_CONDITION.lower()}_alarms_clustered_with_control_actions.xlsx'

print(f"Target: {TARGET_TAG} | Alarm: {ALARM_CONDITION}")
print(f"Events: {EVENTS_FILE}")
print(f"Cluster gap: {CLUSTER_GAP_THRESHOLD} min | Action window: -{ACTION_WINDOW_BEFORE} min / +{ACTION_WINDOW_AFTER} min")
print(f"Trip filtering: {'ON' if FILTER_TRIPS else 'OFF'}")
print(f"Output: {OUTPUT_FILE}")

Target: 03LIC_1016 | Alarm: PVLO
Events: /home/h604827/ControlActions/DATA/25_tags_events_preprocessed/51fda50f-24a3-4632-9803-7ec34744c34b.parquet
Cluster gap: 30 min | Action window: -240 min / +60 min
Trip filtering: ON
Output: ../RESULTS/03LIC_1016_episodes/03LIC_1016_pvlo_alarms_clustered_with_control_actions.xlsx


In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Load events data (supports both CSV and parquet)
if EVENTS_FILE.endswith('.parquet'):
    events_df = pd.read_parquet(EVENTS_FILE)
else:
    events_df = pd.read_csv(EVENTS_FILE, low_memory=False)

events_df['VT_Start'] = pd.to_datetime(events_df['VT_Start'])
events_df = events_df.sort_values('VT_Start').reset_index(drop=True)

# Deduplicate: for rows with same (VT_Start, Source, ConditionName, Description),
# merge them by taking the first non-null value per column across duplicates
dedup_cols = ['VT_Start', 'Source', 'ConditionName', 'Description']
pre_dedup = len(events_df)
events_df = events_df.groupby(dedup_cols, sort=False).first().reset_index()
events_df = events_df.sort_values('VT_Start').reset_index(drop=True)
print(f"Deduplication: {pre_dedup} → {len(events_df)} rows (removed {pre_dedup - len(events_df)} duplicates)")

# ── Trip period filtering ──
if FILTER_TRIPS:
    trips_df = pd.read_csv(TRIP_FILE)
    trips_df['Stop Date'] = pd.to_datetime(trips_df['Stop Date'])
    trips_df['Start Date'] = pd.to_datetime(trips_df['Start Date'])
    
    # Remove events that fall within any trip window (Stop Date = trip start, Start Date = restart)
    pre_trip = len(events_df)
    trip_mask = pd.Series(False, index=events_df.index)
    for _, trip in trips_df.iterrows():
        trip_mask |= (events_df['VT_Start'] >= trip['Stop Date']) & (events_df['VT_Start'] <= trip['Start Date'])
    events_df = events_df[~trip_mask].reset_index(drop=True)
    print(f"Trip filtering: {pre_trip} → {len(events_df)} rows (removed {pre_trip - len(events_df)} during trips)")

# Filter by start date
events_df = events_df[events_df['VT_Start'] >= pd.to_datetime(START_DATE)]

# Filter for target tag alarm events (Category=1 = actual alarm events)
pvlo = events_df[
    (events_df['Source'] == TARGET_TAG) &
    (events_df['ConditionName'] == ALARM_CONDITION) &
    (events_df['Category'] == 1)
].copy()

print(f"\nTarget: {TARGET_TAG} | Condition: {ALARM_CONDITION}")
print(f"Total {ALARM_CONDITION} Category=1 events: {len(pvlo)}")
print(f"  Alarm starts (Action is NaN/blank): {pvlo['Action'].isna().sum()}")
print(f"  Alarm ends (Action = OK): {(pvlo['Action'] == 'OK').sum()}")
print(f"  Date range: {pvlo['VT_Start'].min()} to {pvlo['VT_Start'].max()}")
print(f"\nAction breakdown:")
print(pvlo['Action'].value_counts(dropna=False))

Deduplication: 70213 → 70213 rows (removed 0 duplicates)
Trip filtering: 70213 → 70213 rows (removed 0 during trips)

Target: 03LIC_1016 | Condition: PVLO
Total PVLO Category=1 events: 3926
  Alarm starts (Action is NaN/blank): 1964
  Alarm ends (Action = OK): 1962
  Date range: 2022-01-04 03:44:17.703100 to 2025-06-27 17:32:37.458400

Action breakdown:
Action
None    1964
OK      1962
Name: count, dtype: int64


In [3]:
events_df[events_df['Description'].str.contains(r'MOD*', na=False)]['Description'].value_counts()

Description
MODE                                                                                                  1469
3ER1BD UNIT MOS                                                                                         29
OFFNORM                LOW      3ER1BD UNIT MOS          1E                          1          31      13
MODATTR                                                                                                  2
Name: count, dtype: int64

In [4]:
events_df.columns

Index(['VT_Start', 'Source', 'ConditionName', 'Description', 'Action', 'Actor',
       'AreaName', 'AlarmLimit', 'Block', 'Category', 'EventID', 'Flags',
       'LocalTime', 'LocationFullName', 'LocationTagName', 'PrevValue',
       'Priority', 'ReceivedDelay', 'ServerName', 'ShelvedReason',
       'SourceParameter', 'Station', 'Time', 'TransactionID', 'Units', 'Value',
       'H', 'TagID', 'AlarmStatus'],
      dtype='object')

In [5]:
# Extract alarm episodes by walking through events in order
# Rule: Start = Action is NaN, End = Action is OK
# If we see multiple starts in a row, alarm is still ongoing (PV bouncing near threshold)
# Take the FIRST start followed by the NEXT OK as one episode

episodes_list = []
current_start = None
current_start_value = None

for _, row in pvlo.iterrows():
    is_start = pd.isna(row['Action']) or row['Action'] == ''
    is_end = row['Action'] == 'OK'
    
    if is_start and current_start is None:
        current_start = row['VT_Start']
        current_start_value = row['Value']
    elif is_start and current_start is not None:
        pass  # still in alarm
    elif is_end and current_start is not None:
        episodes_list.append({
            'alarm_start': current_start,
            'alarm_end': row['VT_Start'],
            'start_value': current_start_value,
            'end_value': row['Value'],
        })
        current_start = None
        current_start_value = None

episodes = pd.DataFrame(episodes_list)
episodes['episode_num'] = range(1, len(episodes) + 1)
episodes['duration_minutes'] = (episodes['alarm_end'] - episodes['alarm_start']).dt.total_seconds() / 60
episodes['gap_to_next_minutes'] = (
    episodes['alarm_start'].shift(-1) - episodes['alarm_end']
).dt.total_seconds() / 60

print(f"Total alarm episodes: {len(episodes)}")
print(f"Orphan starts (no matching end): {pvlo['Action'].isna().sum() - len(episodes)}")
print(f"\nAlarm DURATION (minutes):")
print(episodes['duration_minutes'].describe().to_string())
print(f"\nGAP to next alarm (minutes):")
print(episodes['gap_to_next_minutes'].dropna().describe().to_string())

Total alarm episodes: 1959
Orphan starts (no matching end): 5

Alarm DURATION (minutes):
count    1959.000000
mean        7.434283
std       153.032055
min         0.278687
25%         0.739602
50%         1.012427
75%         1.502106
max      5958.278340

GAP to next alarm (minutes):
count      1958.000000
mean        926.999269
std        6352.321751
min           0.599982
25%           2.566719
50%           6.491667
75%          46.922706
max      157827.057992


In [6]:
# Classify alarms by duration
def classify_alarm(duration_min):
    if duration_min <= 1:
        return '≤1 min (fleeting)'
    elif duration_min <= 5:
        return '1-5 min (chattering)'
    elif duration_min <= 30:
        return '5-30 min (short)'
    elif duration_min <= 60:
        return '30-60 min (medium)'
    else:
        return '>60 min (sustained)'

episodes['alarm_class'] = episodes['duration_minutes'].apply(classify_alarm)

class_order = ['≤1 min (fleeting)', '1-5 min (chattering)', '5-30 min (short)', 
               '30-60 min (medium)', '>60 min (sustained)']
class_counts = episodes['alarm_class'].value_counts().reindex(class_order).fillna(0).astype(int)
class_pcts = (class_counts / len(episodes) * 100).round(1)

print("Alarm Duration Classification:")
print("=" * 50)
for cls in class_order:
    print(f"  {cls:25s}: {class_counts[cls]:5d} ({class_pcts[cls]:5.1f}%)")
print(f"  {'TOTAL':25s}: {len(episodes):5d}")

Alarm Duration Classification:
  ≤1 min (fleeting)        :   971 ( 49.6%)
  1-5 min (chattering)     :   865 ( 44.2%)
  5-30 min (short)         :   115 (  5.9%)
  30-60 min (medium)       :     2 (  0.1%)
  >60 min (sustained)      :     6 (  0.3%)
  TOTAL                    :  1959


In [7]:
# Distribution of alarm DURATIONS
fig = make_subplots(rows=2, cols=1, subplot_titles=[
    'Alarm Duration Distribution (all episodes)',
    'Alarm Duration Distribution (zoomed ≤60 min)'
], vertical_spacing=0.12)

fig.add_trace(go.Histogram(x=episodes['duration_minutes'], nbinsx=100, 
                            marker_color='indianred', name='All'), row=1, col=1)
fig.add_trace(go.Histogram(x=episodes[episodes['duration_minutes'] <= 60]['duration_minutes'], 
                            nbinsx=60, marker_color='steelblue', name='≤60 min'), row=2, col=1)

fig.update_xaxes(title_text='Duration (minutes)', row=1, col=1)
fig.update_xaxes(title_text='Duration (minutes)', row=2, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Count', row=2, col=1)
fig.update_layout(height=600, showlegend=False, title_text='How long do alarms last?')
fig.show()

In [8]:
# Distribution of GAP between consecutive alarms
gaps = episodes['gap_to_next_minutes'].dropna()

# Classify gaps
def classify_gap(gap_min):
    if gap_min <= 5:
        return '≤5 min (rapid re-alarm)'
    elif gap_min <= 30:
        return '5-30 min'
    elif gap_min <= 60:
        return '30-60 min'
    elif gap_min <= 360:
        return '1-6 hours'
    elif gap_min <= 1440:
        return '6-24 hours'
    else:
        return '>24 hours'

gap_classes = gaps.apply(classify_gap)
gap_order = ['≤5 min (rapid re-alarm)', '5-30 min', '30-60 min', '1-6 hours', '6-24 hours', '>24 hours']
gap_counts = gap_classes.value_counts().reindex(gap_order).fillna(0).astype(int)
gap_pcts = (gap_counts / len(gaps) * 100).round(1)

print("Gap Between Consecutive Alarms:")
print("=" * 50)
for cls in gap_order:
    print(f"  {cls:25s}: {gap_counts[cls]:5d} ({gap_pcts[cls]:5.1f}%)")

fig = make_subplots(rows=2, cols=1, subplot_titles=[
    'Gap to Next Alarm (all)',
    'Gap to Next Alarm (zoomed ≤120 min)'
], vertical_spacing=0.12)

fig.add_trace(go.Histogram(x=gaps, nbinsx=100, marker_color='darkorange', name='All'), row=1, col=1)
fig.add_trace(go.Histogram(x=gaps[gaps <= 120], nbinsx=60, marker_color='teal', name='≤120 min'), row=2, col=1)

fig.update_xaxes(title_text='Gap to next alarm (minutes)', row=1, col=1)
fig.update_xaxes(title_text='Gap to next alarm (minutes)', row=2, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Count', row=2, col=1)
fig.update_layout(height=600, showlegend=False, title_text='How quickly does the alarm come back?')
fig.show()

Gap Between Consecutive Alarms:
  ≤5 min (rapid re-alarm)  :   875 ( 44.7%)
  5-30 min                 :   490 ( 25.0%)
  30-60 min                :   138 (  7.0%)
  1-6 hours                :   158 (  8.1%)
  6-24 hours               :   136 (  6.9%)
  >24 hours                :   161 (  8.2%)


In [9]:
# Build clusters using configured gap threshold
episodes_all = episodes.copy().reset_index(drop=True)
print(f"Total alarm episodes: {len(episodes_all)}")

# Recompute gaps
episodes_all['gap_to_next_minutes'] = (
    episodes_all['alarm_start'].shift(-1) - episodes_all['alarm_end']
).dt.total_seconds() / 60

# Assign cluster_id: consecutive alarms with gap ≤ threshold belong to the same cluster
cluster_id = 0
cluster_ids = [0]
for g in episodes_all['gap_to_next_minutes'].iloc[:-1]:
    if pd.notna(g) and g <= CLUSTER_GAP_THRESHOLD:
        cluster_ids.append(cluster_id)
    else:
        cluster_id += 1
        cluster_ids.append(cluster_id)

episodes_all['cluster_id'] = cluster_ids

# Compute cluster-level stats
clusters = episodes_all.groupby('cluster_id').agg(
    cluster_start=('alarm_start', 'min'),
    cluster_end=('alarm_end', 'max'),
    n_alarms=('episode_num', 'count')
).sort_values('cluster_start')

clusters['total_duration_min'] = (clusters['cluster_end'] - clusters['cluster_start']).dt.total_seconds() / 60
clusters['gap_to_next_cluster_min'] = (
    clusters['cluster_start'].shift(-1) - clusters['cluster_end']
).dt.total_seconds() / 60

# Classify cluster types
def classify_cluster(row):
    if row['n_alarms'] == 1 and row['total_duration_min'] <= 5:
        return 'Isolated brief alarm'
    elif row['n_alarms'] <= 3 and row['total_duration_min'] <= 30:
        return 'Small cluster'
    elif row['total_duration_min'] <= 120:
        return 'Medium situation (<2h)'
    else:
        return 'Extended situation (>2h)'

clusters['cluster_type'] = clusters.apply(classify_cluster, axis=1)

# Summary
print(f"\n{len(episodes_all)} raw alarms → {len(clusters)} independent clusters (gap threshold: {CLUSTER_GAP_THRESHOLD} min)")
print(f"  Single-alarm: {(clusters['n_alarms'] == 1).sum()}")
print(f"  Multi-alarm: {(clusters['n_alarms'] > 1).sum()}")
print(f"\nCluster sizes (# alarms per cluster):")
print(clusters['n_alarms'].describe().to_string())
print(f"\nGap between clusters (minutes):")
print(clusters['gap_to_next_cluster_min'].dropna().describe().to_string())
print(f"\nCluster types:")
for ctype in ['Isolated brief alarm', 'Small cluster', 'Medium situation (<2h)', 'Extended situation (>2h)']:
    count = (clusters['cluster_type'] == ctype).sum()
    pct = count / len(clusters) * 100
    avg_alarms = clusters[clusters['cluster_type'] == ctype]['n_alarms'].mean()
    avg_dur = clusters[clusters['cluster_type'] == ctype]['total_duration_min'].mean()
    print(f"  {ctype:30s}: {count:4d} ({pct:5.1f}%) | avg {avg_alarms:.1f} alarms, avg {avg_dur:.0f} min")

Total alarm episodes: 1959

1959 raw alarms → 594 independent clusters (gap threshold: 30 min)
  Single-alarm: 378
  Multi-alarm: 216

Cluster sizes (# alarms per cluster):
count    594.00000
mean       3.29798
std        9.29707
min        1.00000
25%        1.00000
50%        1.00000
75%        2.00000
max      140.00000

Gap between clusters (minutes):
count       593.000000
mean       3046.268345
std       11266.760157
min          30.008337
25%          66.834158
50%         368.965018
75%        1650.232512
max      157827.057992

Cluster types:
  Isolated brief alarm          :  352 ( 59.3%) | avg 1.0 alarms, avg 2 min
  Small cluster                 :  125 ( 21.0%) | avg 2.0 alarms, avg 12 min
  Medium situation (<2h)        :   91 ( 15.3%) | avg 6.3 alarms, avg 55 min
  Extended situation (>2h)      :   26 (  4.4%) | avg 30.4 alarms, avg 620 min


In [10]:
# Distribution of gaps between clusters
cluster_gaps = clusters['gap_to_next_cluster_min'].dropna()

def classify_cluster_gap(gap_min):
    if gap_min <= 60:
        return '≤1 hour'
    elif gap_min <= 360:
        return '1-6 hours'
    elif gap_min <= 1440:
        return '6-24 hours'
    elif gap_min <= 4320:
        return '1-3 days'
    elif gap_min <= 10080:
        return '3-7 days'
    else:
        return '>7 days'

gap_cls = cluster_gaps.apply(classify_cluster_gap)
gap_order = ['≤1 hour', '1-6 hours', '6-24 hours', '1-3 days', '3-7 days', '>7 days']
gap_counts = gap_cls.value_counts().reindex(gap_order).fillna(0).astype(int)
gap_pcts = (gap_counts / len(cluster_gaps) * 100).round(1)

print(f"Gap Between Alarm Clusters ({CLUSTER_GAP_THRESHOLD}-min threshold, {TARGET_TAG}):")
print("=" * 60)
for cls in gap_order:
    print(f"  {cls:20s}: {gap_counts[cls]:5d} ({gap_pcts[cls]:5.1f}%)")

fig = make_subplots(rows=2, cols=1, subplot_titles=[
    f'Gap Between Clusters — All ({TARGET_TAG})',
    f'Gap Between Clusters — Zoomed ≤1440 min / 24h ({TARGET_TAG})'
], vertical_spacing=0.15)

fig.add_trace(go.Histogram(x=cluster_gaps, nbinsx=80, marker_color='mediumpurple'), row=1, col=1)
fig.add_trace(go.Histogram(x=cluster_gaps[cluster_gaps <= 1440], nbinsx=60, marker_color='mediumseagreen'), row=2, col=1)

fig.update_xaxes(title_text='Gap (minutes)', row=1, col=1)
fig.update_xaxes(title_text='Gap (minutes)', row=2, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Count', row=2, col=1)
fig.update_layout(height=600, showlegend=False, 
                  title_text=f'Gap between alarm clusters ({CLUSTER_GAP_THRESHOLD}-min threshold, {TARGET_TAG})')
fig.show()

Gap Between Alarm Clusters (30-min threshold, 03LIC_1016):
  ≤1 hour             :   138 ( 23.3%)
  1-6 hours           :   158 ( 26.6%)
  6-24 hours          :   136 ( 22.9%)
  1-3 days            :    76 ( 12.8%)
  3-7 days            :    50 (  8.4%)
  >7 days             :    35 (  5.9%)


In [11]:
# Try multiple gap thresholds to find truly independent alarm situations
for threshold in [5, 15, 30, 60, 120]:
    cid = 0
    cids = [0]
    for g in episodes_all['gap_to_next_minutes'].iloc[:-1]:
        if pd.notna(g) and g <= threshold:
            cids.append(cid)
        else:
            cid += 1
            cids.append(cid)
    n_clusters = cid + 1
    # Compute inter-cluster gaps for this threshold
    episodes_all[f'_cid_{threshold}'] = cids
    cd = episodes_all.groupby(f'_cid_{threshold}').agg(
        start=('alarm_start', 'min'), end=('alarm_end', 'max'),
        n_alarms=('episode_num', 'count')
    ).sort_values('start')
    cd['gap_next'] = (cd['start'].shift(-1) - cd['end']).dt.total_seconds() / 60
    median_gap = cd['gap_next'].dropna().median()
    pct_under_30 = (cd['gap_next'].dropna() <= 30).mean() * 100
    
    print(f"Threshold {threshold:>3d} min → {n_clusters:>4d} clusters | "
          f"median inter-cluster gap: {median_gap:>8.1f} min | "
          f"% clusters with gap ≤30 min: {pct_under_30:.1f}%")

Threshold   5 min → 1084 clusters | median inter-cluster gap:     35.7 min | % clusters with gap ≤30 min: 45.2%
Threshold  15 min →  754 clusters | median inter-cluster gap:    108.4 min | % clusters with gap ≤30 min: 21.2%
Threshold  30 min →  594 clusters | median inter-cluster gap:    369.0 min | % clusters with gap ≤30 min: 0.0%
Threshold  60 min →  456 clusters | median inter-cluster gap:   1141.7 min | % clusters with gap ≤30 min: 0.0%
Threshold 120 min →  361 clusters | median inter-cluster gap:   1321.7 min | % clusters with gap ≤30 min: 0.0%


In [12]:
# Build output dataframe: one row per raw alarm, with cluster info
output = episodes_all[['episode_num', 'alarm_start', 'alarm_end', 'duration_minutes', 
                         'gap_to_next_minutes', 'start_value', 'end_value', 'cluster_id']].copy()

# Add cluster-level info
output = output.merge(
    clusters[['n_alarms', 'cluster_start', 'cluster_end', 'total_duration_min', 'gap_to_next_cluster_min', 'cluster_type']],
    left_on='cluster_id', right_index=True
)
output = output.rename(columns={
    'n_alarms': 'cluster_total_alarms',
    'cluster_start': 'cluster_start_time',
    'cluster_end': 'cluster_end_time',
    'total_duration_min': 'cluster_total_duration_min'
})

# Renumber clusters from 1
cluster_id_map = {old: new for new, old in enumerate(sorted(output['cluster_id'].unique()), 1)}
output['cluster_id'] = output['cluster_id'].map(cluster_id_map)
output = output.sort_values('alarm_start').reset_index(drop=True)

# --- Extract operator control actions (CHANGE events) per cluster ---
WINDOW_BEFORE = pd.Timedelta(minutes=ACTION_WINDOW_BEFORE)
WINDOW_AFTER = pd.Timedelta(minutes=ACTION_WINDOW_AFTER)

change_events = events_df[
    events_df['ConditionName'] == 'CHANGE'
].copy()

# Build cluster lookup with renumbered IDs
cluster_lookup = clusters.copy()
cluster_lookup['cluster_id_new'] = cluster_lookup.index.map(cluster_id_map)
cluster_lookup = cluster_lookup.sort_values('cluster_start').reset_index(drop=True)

actions_list = []
for _, cl in cluster_lookup.iterrows():
    window_start = cl['cluster_start'] - WINDOW_BEFORE
    window_end = cl['cluster_end'] + WINDOW_AFTER
    
    mask = (change_events['VT_Start'] >= window_start) & (change_events['VT_Start'] <= window_end)
    cluster_actions = change_events[mask].copy()
    cluster_actions['cluster_id'] = cl['cluster_id_new']
    cluster_actions['cluster_start'] = cl['cluster_start']
    cluster_actions['cluster_end'] = cl['cluster_end']
    actions_list.append(cluster_actions)

control_actions = pd.concat(actions_list, ignore_index=True)
control_actions = control_actions.sort_values(['cluster_id', 'VT_Start']).reset_index(drop=True)

# Classify action timing relative to cluster duration
control_actions['action_timing'] = np.where(
    control_actions['VT_Start'] < control_actions['cluster_start'], 'before',
    np.where(control_actions['VT_Start'] > control_actions['cluster_end'], 'after', 'during')
)

# Classify action direction based on Value vs PrevValue
_val_num = pd.to_numeric(control_actions['Value'], errors='coerce')
_prev_num = pd.to_numeric(control_actions['PrevValue'], errors='coerce')
control_actions['action_direction'] = np.where(
    _val_num > _prev_num, 'increase',
    np.where(_val_num < _prev_num, 'decrease', 'no_change')
)

print(f"Control actions extracted: {len(control_actions)} CHANGE events across {control_actions['cluster_id'].nunique()} clusters")
print(f"Clusters with no actions: {output['cluster_id'].nunique() - control_actions['cluster_id'].nunique()}")
print(f"\nAction timing breakdown:")
print(control_actions['action_timing'].value_counts().to_string())
print(f"\nTop 10 most acted-on tags:")
print(control_actions['Source'].value_counts().head(10).to_string())

# Save both sheets to Excel
actions_save_cols = ['cluster_id', 'cluster_start', 'cluster_end', 'action_timing', 'action_direction', 'Source', 'Description', 'VT_Start', 'PrevValue', 'Value']
with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    output.to_excel(writer, sheet_name='alarm_clusters', index=False)
    control_actions[actions_save_cols].to_excel(writer, sheet_name='control_actions', index=False)

print(f"\nSaved to {OUTPUT_FILE}")
print(f"  Sheet 'alarm_clusters': {len(output)} alarms across {output['cluster_id'].nunique()} clusters")
print(f"  Sheet 'control_actions': {len(control_actions)} actions")
output.head(10)

Control actions extracted: 3729 CHANGE events across 296 clusters
Clusters with no actions: 298

Action timing breakdown:
action_timing
before    2327
during     817
after      585

Top 10 most acted-on tags:
Source
03LIC_1034      2106
03LIC_1016       900
03TIC_1009       247
03LIC_1031       176
03HIC_1009A       73
04RES_3ER1BD      60
03HIC_1023A       53
03TIC_1023        39
03HIC_1009B       25
03HIC_1023B       20

Saved to ../RESULTS/03LIC_1016_episodes/03LIC_1016_pvlo_alarms_clustered_with_control_actions.xlsx
  Sheet 'alarm_clusters': 1959 alarms across 594 clusters
  Sheet 'control_actions': 3729 actions


,episode_num,alarm_start,alarm_end,duration_minutes,gap_to_next_minutes,start_value,end_value,cluster_id,cluster_total_alarms,cluster_start_time,cluster_end_time,cluster_total_duration_min,gap_to_next_cluster_min,cluster_type
0,1,2022-01-04 03:44:17.703100,2022-01-04 03:44:41.702000,0.399982,931.082530,28.699,31.832,1,1,2022-01-04 03:44:17.703100,2022-01-04 03:44:41.702000,0.399982,931.082530,Isolated brief alarm
1,2,2022-01-04 19:15:46.653800,2022-01-04 19:16:18.654500,0.533345,1.854128,28.375,31.811,2,2,2022-01-04 19:15:46.653800,2022-01-04 19:18:34.652700,2.799982,576.720015,Small cluster
2,3,2022-01-04 19:18:09.902200,2022-01-04 19:18:34.652700,0.412508,576.720015,28.720,31.981,2,2,2022-01-04 19:15:46.653800,2022-01-04 19:18:34.652700,2.799982,576.720015,Small cluster
3,4,2022-01-05 04:55:17.853600,2022-01-05 04:55:44.854000,0.450007,1.666667,28.563,31.980,3,7,2022-01-05 04:55:17.853600,2022-01-05 05:19:51.652500,24.563315,203.286683,Medium situation (<2h)
4,5,2022-01-05 04:57:24.854000,2022-01-05 04:58:16.855200,0.866687,4.050018,28.518,31.836,3,7,2022-01-05 04:55:17.853600,2022-01-05 05:19:51.652500,24.563315,203.286683,Medium situation (<2h)
5,6,2022-01-05 05:02:19.856300,2022-01-05 05:03:13.854100,0.899963,4.054170,28.224,32.073,3,7,2022-01-05 04:55:17.853600,2022-01-05 05:19:51.652500,24.563315,203.286683,Medium situation (<2h)
6,7,2022-01-05 05:07:17.104300,2022-01-05 05:08:07.854900,0.845843,4.029958,28.718,31.803,3,7,2022-01-05 04:55:17.853600,2022-01-05 05:19:51.652500,24.563315,203.286683,Medium situation (<2h)
7,8,2022-01-05 05:12:09.652400,2022-01-05 05:12:58.652600,0.816670,4.300057,28.331,31.991,3,7,2022-01-05 04:55:17.853600,2022-01-05 05:19:51.652500,24.563315,203.286683,Medium situation (<2h)
8,9,2022-01-05 05:17:16.656000,2022-01-05 05:17:34.652400,0.299940,1.920822,28.694,31.867,3,7,2022-01-05 04:55:17.853600,2022-01-05 05:19:51.652500,24.563315,203.286683,Medium situation (<2h)
9,10,2022-01-05 05:19:29.901700,2022-01-05 05:19:51.652500,0.362513,203.286683,28.682,31.822,3,7,2022-01-05 04:55:17.853600,2022-01-05 05:19:51.652500,24.563315,203.286683,Medium situation (<2h)


In [27]:
output['gap_to_next_cluster_min'].describe()

count      1378.000000
mean       3086.578978
std       10579.062780
min          30.429168
25%         106.718575
50%         627.696683
75%        2651.154967
max      131424.021448
Name: gap_to_next_cluster_min, dtype: float64

In [ ]:
# Control action stats for top acted-on tags (SP/OP only)
control_actions['change_magnitude'] = (pd.to_numeric(control_actions['Value'], errors='coerce') - pd.to_numeric(control_actions['PrevValue'], errors='coerce')).abs()

# Filter to SP/OP description events only
sp_op_actions = control_actions[control_actions['Description'].isin(['SP', 'OP'])].copy()
print(f"Total control actions: {len(control_actions)} → SP/OP only: {len(sp_op_actions)}")

# Analyze top tags + the target tag itself
top_tags = sp_op_actions['Source'].value_counts().head(5).index.tolist()
target_tags = list(dict.fromkeys([TARGET_TAG] + top_tags))  # target first, then top, deduplicated

for tag in target_tags:
    tag_actions = sp_op_actions[sp_op_actions['Source'] == tag]
    if len(tag_actions) == 0:
        continue
    n_prevval_missing = tag_actions['PrevValue'].isna().sum()
    print(f"\n{'='*60}")
    print(f"Tag: {tag} — {len(tag_actions)} SP/OP actions (PrevValue missing: {n_prevval_missing})")
    print(f"{'='*60}")
    
    for desc in ['OP', 'SP']:
        desc_actions = tag_actions[tag_actions['Description'] == desc]
        if len(desc_actions) == 0:
            continue
        print(f"\n  [{desc}] — {len(desc_actions)} actions")
        for d in ['increase', 'decrease', 'no_change']:
            subset = desc_actions[desc_actions['action_direction'] == d]
            n = len(subset)
            if n == 0:
                print(f"    {d}: 0 actions")
                continue
            mag = subset['change_magnitude']
            print(f"    {d}: {n} actions | "
                  f"magnitude — mean: {mag.mean():.2f}, median: {mag.median():.2f}, "
                  f"min: {mag.min():.2f}, max: {mag.max():.2f}, std: {mag.std():.2f}")

Total control actions: 16094 → SP/OP only: 15128

Tag: 03LIC_1071 — 1075 SP/OP actions (PrevValue missing: 0)

  [OP] — 794 actions
    increase: 408 actions | magnitude — mean: 2.78, median: 2.00, min: 0.10, max: 50.00, std: 4.57
    decrease: 357 actions | magnitude — mean: 3.76, median: 2.00, min: 0.10, max: 50.00, std: 7.05
    no_change: 29 actions | magnitude — mean: nan, median: nan, min: nan, max: nan, std: nan

  [SP] — 281 actions
    increase: 189 actions | magnitude — mean: 2.09, median: 2.00, min: 0.01, max: 35.00, std: 3.82
    decrease: 92 actions | magnitude — mean: 3.81, median: 2.00, min: 0.87, max: 34.27, std: 5.44
    no_change: 0 actions

Tag: 03LIC_1016 — 498 SP/OP actions (PrevValue missing: 0)

  [OP] — 335 actions
    increase: 147 actions | magnitude — mean: 4.01, median: 2.00, min: 0.50, max: 30.00, std: 4.32
    decrease: 90 actions | magnitude — mean: 8.04, median: 2.00, min: 0.56, max: 72.78, std: 11.44
    no_change: 98 actions | magnitude — mean: 0.00, m

In [29]:
# Check if any SP/OP rows (all tags) still have PrevValue missing after dedup fix
sp_op_all = control_actions[control_actions['Description'].isin(['SP', 'OP'])]
missing_pv = sp_op_all[sp_op_all['PrevValue'].isna()]

print(f"SP/OP rows with PrevValue missing: {len(missing_pv)} / {len(sp_op_all)} total")
print(f"\nBreakdown by Source:")
print(missing_pv['Source'].value_counts().head(20).to_string())

if len(missing_pv) > 0:
    # Cross-check against raw data
    raw_events = pd.read_csv("../DATA/trip_filtered_events.csv", low_memory=False)
    raw_events['VT_Start'] = pd.to_datetime(raw_events['VT_Start'])
    raw_events['PrevValue'] = pd.to_numeric(raw_events['PrevValue'], errors='coerce')
    raw_change = raw_events[raw_events['ConditionName'] == 'CHANGE']

    found_in_raw = 0
    sample_printed = 0
    for _, row in missing_pv.iterrows():
        raw_matches = raw_change[
            (raw_change['Source'] == row['Source']) &
            (raw_change['VT_Start'] == row['VT_Start']) &
            (raw_change['Description'] == row['Description'])
        ]
        if raw_matches['PrevValue'].notna().any():
            found_in_raw += 1
            if sample_printed < 10:
                print(f"  STILL FOUND: {row['Source']} | {row['Description']} | {row['VT_Start']}")
                sample_printed += 1

    print(f"\nSummary: {found_in_raw} / {len(missing_pv)} missing-PrevValue rows have PrevValue in raw data")

SP/OP rows with PrevValue missing: 0 / 15128 total

Breakdown by Source:
Series([], )


In [30]:
control_actions[control_actions['Description'] == 'MODE']['Value'].value_counts()

Value
MAN       401
NORMAL    263
AUTO      122
CAS        59
Name: count, dtype: int64

In [31]:
control_actions[control_actions['Description'] == 'MODE']['PrevValue'].value_counts()

PrevValue
MAN     430
CAS     239
AUTO    176
Name: count, dtype: int64